In [14]:
%load_ext autoreload
%autoreload 
%reload_ext autoreload

import sys
sys.path.append("..")
from src.features.building_height import *
from src.features.nested_functions import *

import fiona
import pandas as pd
import geopandas as gpd
import os
from datetime import datetime


pd.set_option('display.max_columns', None)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Run building height process

This assumes the pre-processing steps below have already been run.

In [6]:
#define town names
mmc_town_names = ['Arlington', 'Boston', 'Braintree', 'Brookline', 'Cambridge', 'Chelsea', 
             'Everett', 'Lynn', 'Malden', 'Medford', 'Melrose', 'Newton', 'Quincy', 'Revere', 
             'Somerville', 'Watertown', 'Winthrop']

#read in cool roofs roofprint
cool_roofs_fp = r'K:\DataServices\Projects\Current_Projects\Climate_Change\MVP_MMC_CoolRoofs_MVP\Data\Analysis_Data\Data_Cool_Roofs\2_Output\MMC_Cool_Roofs.shp'
cool_roofs_gdf = gpd.read_file(cool_roofs_fp)

#set directory path
path = r"\\data-sync\public\DataServices\Projects\Current_Projects"


In [15]:
gdb_path = os.path.join(path, 'Neighborhood_Planning_and_Zoning\Zoning_Projects\Rightsizing_Zoning_2025\RightsizingZoning_2025.gdb')
output_layer_name = '00_mmc_enriched_structures'
list_of_town_names = mmc_town_names

mmc_footprints_with_height_and_flat = run_building_height_process(gdb_path=gdb_path, 
                                                                list_of_town_names=list_of_town_names,
                                                                output_layer_name=output_layer_name,
                                                                cool_roofs_gdf=cool_roofs_gdf)


Reading in and merging footprints with building heights, created from NDSM
Reading in parcel database


c:\Users\RBowers\rb-scripting\rightsizing-zoning2025\notebooks\..\src\features\building_height.py:339: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_towns_parcels = pd.concat([all_towns_parcels, muni_parcels])
c:\Users\RBowers\AppData\Local\ESRI\conda\envs\geospatial-env\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\RBowers\rb-scripting\rightsizing-zoning2025\notebooks\..\src\features\nested_functions.py:319: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  shu

Joining roofprints with parcels
Joining with cool roofs data
Calculating primary structure
Exporting to project geodatabase


c:\Users\RBowers\AppData\Local\ESRI\conda\envs\geospatial-env\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered layer name: '00_mmc_enriched_structures' to '_00_mmc_enriched_structures_1_2_3'
  ogr_write(
c:\Users\RBowers\AppData\Local\ESRI\conda\envs\geospatial-env\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field primary_structure of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(


## Run ArcPy Lidar pre-processing

raster functions to create ndsm rasters for buildings in each MMC town (bounding box).

*only run again if necessary - takes a while and ndsm has been run for each town in MMC already*


Town names

In [4]:
mmc_town_names = ['Arlington', 'Boston', 'Braintree', 'Brookline', 'Cambridge', 'Chelsea', 
             'Everett', 'Lynn', 'Malden', 'Medford', 'Melrose', 'Newton', 'Quincy', 'Revere', 
             'Somerville', 'Watertown', 'Winthrop']


In [ ]:
#run again only if need be!

#gis = GIS("https://metroboston.maps.arcgis.com/portal", "rbowers_metroboston", "e8t7iRf1M6a3")
#arcpy.SignInToPortal("https://metroboston.maps.arcgis.com/","rbowers_metroboston","e8t7iRf1M6a3")

las_folder = 'I:\Imagery\MassGIS_LAS_files'

for town_name in mmc_town_names:
    
    print(town_name + ' processing starting at ' + str(datetime.now()))

    #create las dataset 
    las_dataset = create_las_dataset(town_name = town_name, 
                                     las_folder=las_folder) 
    
    #create an ndsm raster
    ndsm_raster = create_ndsm_raster(town_name=town_name,
                                    las_dataset=las_dataset)
    


Lynn processing starting at 2025-06-26 10:44:25.585329
dtm layer creation
dsm layer creation
ndsm layer creation


this is code to get las files for lynn - can do something wtih this later!

In [ ]:
#FIRST, DOWNLAOD LAZ FILES FROM NOAA AND ADD TO DOWNLOAD FOLDER

import urllib.request 

arcpy.env.outputCoordinateSystem = arcpy.SpatialReference("NAD 1983 StatePlane Massachusetts FIPS 2001 (Meters)")
env.overwriteOutput = True

town_name = 'Lynn'

muni_gdf = munis.loc[munis[muni_field].str.casefold() == town_name.casefold()]
index_fp = r'I:\Imagery\MassGIS_LAS_files\goodies\goodies\indices\USGS_MA_CentralEastern_1_2021_TileIndex.shp'
index = gpd.read_file(index_fp)

#first, reproject all to mass mainland
mass_mainland_crs = "EPSG:26986"

index = index.to_crs(mass_mainland_crs)
muni_gdf = muni_gdf.to_crs(mass_mainland_crs)

#then, use spatial join to identify intersection between muni boundary and tiles 
intersecting = index.sjoin(muni_gdf, how='inner')
intersecting_list =  intersecting['Tile_ID'].tolist()

#use text file from NOAA to download laz files and convert to las
download_txt = r"I:\Imagery\MassGIS_LAS_files\goodies\goodies\0_file_download_links.txt"

file_list = []

for item in intersecting_list: #for each las tile identified by the intersect
    filename = "I:\\Imagery\\MassGIS_LAS_files\\laz\\" + item + ".laz"
    with open(download_txt) as f: #open url list
        for url in f:
            if item in url: #if there's a match between url and tile code, download
                urllib.request.urlretrieve(url, filename)
    file_list.append(filename)
    #now convert to las
print(file_list)

#THEN, CONVERT LAZ TO LAS - do this in ArcPro (convert LAS, link to .laz folder to batch convert)


['I:\\Imagery\\MassGIS_LAS_files\\19TCH336708.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH334707.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH336707.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH337707.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH334705.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH336705.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH337705.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH339705.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH340705.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH334704.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH336704.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH337704.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH339704.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH340704.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH336702.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH337702.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH339702.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH340702.laz', 'I:\\Imagery\\MassGIS_LAS_files\\19TCH342702.laz', 'I:\\Imagery\\MassGIS_LAS_file

Finally, put all of the footprint layers together and enrich with "stories" (meters / 3.3), land parcel info, and "flat_roof" field from Cool Roofs layer. Exports to a gdb feature layer and returns a geodataframe.

Run building height function to do zonal statistics - for each town, generates a building footprint layer with summary statistics for raster cells within each footprint boundary

*only run again if necessary - takes a while and  has been run for each town in MMC already*

In [3]:
from src.features.building_height import *

#only run again if necessary
mmc_town_names = ['Lynn']

for town_name in mmc_town_names:
    
    print(town_name + ' processing starting at ' + str(datetime.now()))

    #RUN BUILDING HEIGHT FUNCTION
    stories = get_building_heights_from_ndsm(town_name = town_name) 

Lynn processing starting at 2025-06-26 11:11:42.394888
